In [29]:
import pandas as pd
import numpy as np

In [30]:
df = pd.read_csv('data_extraction.csv')

In [31]:
df

,Unnamed: 0,pos_x,pscale
0,0,0.000000,1.000000
1,1,0.342143,0.948890
2,2,0.684286,0.897778
3,3,1.026429,0.846667
4,4,1.368571,0.795556
5,5,1.710714,0.744445
6,6,2.052857,0.693334
7,7,2.395000,0.642223
8,8,2.737143,0.591111
9,9,3.079286,0.540000


In [32]:
df = df.drop("Unnamed: 0",axis=1)

In [33]:
from sklearn.linear_model import LinearRegression
#Prepare data
X_POS = np.arange(len(df)).reshape(-1,1)
Y_POS = df['pos_x'].values
X_POS, Y_POS


(array([[ 0],
        [ 1],
        [ 2],
        [ 3],
        [ 4],
        [ 5],
        [ 6],
        [ 7],
        [ 8],
        [ 9],
        [10],
        [11],
        [12],
        [13],
        [14]]),
 array([0.        , 0.34214285, 0.6842857 , 1.02642858, 1.3685714 ,
        1.71071434, 2.05285716, 2.39499998, 2.7371428 , 3.07928562,
        3.42142868, 3.7635715 , 4.10571432, 4.44785738, 4.78999996]))

In [61]:
import pickle
model_pos = LinearRegression()
model_pos.fit(X_POS, Y_POS)
next_pos_x = model_pos.predict([[len(df)]])[0]
next_pos_x

with open("model_pos.pkl","wb") as f:
  pickle.dump(model_pos,f)

In [62]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

poly = PolynomialFeatures(degree=3)
X_PSCALE = poly.fit_transform(np.arange(len(df)).reshape(-1,1))
Y_PSCALE = df["pscale"].values
model_pscale = LinearRegression()
model_pscale.fit(X_PSCALE,Y_PSCALE)
next_pscale = model_pscale.predict(poly.transform(([[len(df)]])))[0]

with open("model_pscale.pkl","wb") as f:
  pickle.dump(model_pscale,f)

In [36]:
next_pscale

np.float64(0.23333353378396227)

In [37]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
from sklearn.preprocessing import MinMaxScaler


In [38]:
scaler = MinMaxScaler()
scaled = scaler.fit_transform(df)
scaled

array([[0.        , 1.        ],
       [0.07142857, 0.92857244],
       [0.14285714, 0.85714371],
       [0.21428572, 0.78571515],
       [0.28571428, 0.7142865 ],
       [0.35714287, 0.64285785],
       [0.42857144, 0.57142921],
       [0.5       , 0.50000048],
       [0.57142856, 0.42857192],
       [0.64285713, 0.35714319],
       [0.71428574, 0.28571458],
       [0.78571431, 0.21428594],
       [0.85714287, 0.14285729],
       [0.92857149, 0.07142865],
       [1.        , 0.        ]])

In [39]:
def create_sequences(data,seq_length):
  x,y=[],[]
  for i in range(len(data)-seq_length):
    x.append(data[i:(i+seq_length)])
    y.append(data[i+seq_length])
  return np.array(x),np.array(y)

In [41]:
sequence_length = 3
x,y = create_sequences(scaled,sequence_length)

In [45]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
model = Sequential()
model.add(LSTM(50, activation='relu',input_shape=(x.shape[1],x.shape[2])))
model.add(Dense(2))

model.compile(optimizer='adam',loss='mse')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [46]:
model.fit(x,y,epochs=200,verbose=1)

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - loss: 0.2617
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - loss: 0.2554
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - loss: 0.2491
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 0.2429
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 0.2366
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - loss: 0.2304
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - loss: 0.2242
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - loss: 0.2180
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 306ms/step - loss: 0.2119
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - loss: 0.2058
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - loss: 0.1996
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - loss: 0.1935
Epoch 13/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step - loss: 0.1874
Epoch 14/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step - loss: 0.1812
Epoch 15/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - loss: 0.1751
Epoch 16/200


In [50]:
last_seq = x[-1:]
predicted = model.predict(last_seq)

predicted_original_scale = scaler.inverse_transform(predicted)
print("Predicted next position:",predicted_original_scale[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
Predicted next position: [4.822714  0.2950645]


In [51]:
df

,pos_x,pscale
0,0.000000,1.000000
1,0.342143,0.948890
2,0.684286,0.897778
3,1.026429,0.846667
4,1.368571,0.795556
5,1.710714,0.744445
6,2.052857,0.693334
7,2.395000,0.642223
8,2.737143,0.591111
9,3.079286,0.540000


In [58]:
#recursive pred
num_predictions = 5
predictions = []
last_sequence = x[-1:]
for _ in range(num_predictions):
  predicted = model.predict(last_sequence)
  predictions.append(predicted[0])
  predicted_reshaped = predicted.reshape(1,1,2)
  last_sequence = np.concatenate((last_sequence[:,1:,:],predicted_reshaped),axis=1)

predictions = scaler.inverse_transform(predictions)
print("Predictions:",predictions)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
Predictions: [[4.82271384 0.29506453]
 [5.12151554 0.27114856]
 [5.43588704 0.25261269]
 [5.76959816 0.23752305]
 [6.09838887 0.22377041]]


In [60]:
df.tail()

,pos_x,pscale
10,3.421429,0.488889
11,3.763572,0.437778
12,4.105714,0.386667
13,4.447857,0.335556
14,4.790000,0.284444
